# Federated Learning for Lung Cancer Histopathology Images with PIDL

This notebook runs the **existing ResNet-18 + PIDL + federated learning pipeline** on the **lung cancer** subset of the Kaggle dataset `andrewmvd/lung-and-colon-cancer-histopathological-images`.

- The **model, loss, and FL loop** are exactly the same as in the brain tumor and colon cancer projects.
- We only change **which dataset path we use** and **how many clients** we simulate.
- Uses **3 clients** with **true secure aggregation (SecAgg+)** and **no DP noise** (DP noise disabled).
- Results (metrics, timings, confusion matrix, summary, and final model weights) are written into **dataset-specific directories**, so brain, colon, and lung runs do not mix.


In [ ]:
# Mount Google Drive (optional, e.g., if you want to copy artifacts to Drive)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone the repository (use your GitHub URL) and go into project root
# If you already have the repo under /content, skip clone and set PROJECT_DIR accordingly.
import os
REPO_URL = "https://github.com/PulockDas/brain-tumor-classification-PIDL-FL.git"  # ← set your repo URL if different
PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"

if not os.path.isdir(PROJECT_DIR):
    !git clone --depth 1 {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}


In [ ]:
# Install project and dependencies (including kagglehub for dataset download)
!cd {PROJECT_DIR} && pip install -e . kagglehub


In [ ]:
# Verify GPU availability and setup
import torch

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

if torch.cuda.is_available():
    print(f"✅ GPU is AVAILABLE!")
    print(f"   GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
    print(f"\n✅ Training will use GPU (much faster than CPU)")
    
    # Test GPU with a simple operation
    test_tensor = torch.randn(1000, 1000).cuda()
    _ = test_tensor @ test_tensor
    print(f"✅ GPU computation test: PASSED")
else:
    print("⚠️  GPU is NOT AVAILABLE - Training will use CPU (very slow!)")
    print("\n📝 To enable GPU in Google Colab:")
    print("   1. Go to: Runtime → Change runtime type")
    print("   2. Set Hardware accelerator: GPU (T4 or better)")
    print("   3. Click Save and re-run this cell")
    print("\n⚠️  WARNING: CPU training will be VERY slow (10-100x slower)")

print("=" * 60)

## Download lung cancer dataset from Kaggle


In [ ]:
import os
import kagglehub

# Download or reuse cached LC25000 lung & colon dataset
kaggle_path = kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images")
print("Kaggle dataset base path:", kaggle_path)

# Expected structure inside kaggle_path:
# lung_colon_image_set/
#   colon_image_sets/
#       colon_aca/
#       colon_n/
#   lung_image_sets/
#       lung_aca/
#       lung_scc/
#       lung_n/

lung_root = os.path.join(kaggle_path, "lung_colon_image_set", "lung_image_sets")
print("Lung dataset root:", lung_root)

if not os.path.isdir(lung_root):
    raise FileNotFoundError(f"Lung dataset root not found: {lung_root}")

print("Lung classes:", os.listdir(lung_root))


## Configuration


In [ ]:
import os

# Paths
PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"

# Dataset configuration
DATA_ROOT = lung_root           # Folder containing class subfolders (lung_aca, lung_scc, lung_n)
DATASET_NAME = "lung_cancer"   # Used only for naming result directories

# Federated learning configuration
NUM_ROUNDS = 10                  # Number of FL rounds (change as needed)
LOCAL_EPOCHS = 5                 # Local epochs per round
NUM_CLIENTS = 3                  # 3 clients

# Differential Privacy configuration
DP_NOISE_FRACTION = 0.0          # DP noise disabled (0% of parameters to add noise to)
DP_NOISE_SCALE = 0.01            # DP noise standard deviation (not used when fraction is 0)

# Dataset- and client-specific log directory structure
LOG_DIR = os.path.join(
    "/content/results_lung", DATASET_NAME, f"{NUM_CLIENTS}_clients"
)

# Determine number of classes from dataset
num_classes = len(os.listdir(DATA_ROOT))

print("DATA_ROOT         :", DATA_ROOT)
print("LOG_DIR           :", LOG_DIR)
print("DATASET_NAME      :", DATASET_NAME)
print("NUM_CLASSES       :", num_classes)
print("NUM_ROUNDS        :", NUM_ROUNDS)
print("LOCAL_EPOCHS      :", LOCAL_EPOCHS)
print("NUM_CLIENTS       :", NUM_CLIENTS)
print("DP_NOISE_FRACTION :", DP_NOISE_FRACTION)
print("DP_NOISE_SCALE    :", DP_NOISE_SCALE)


## Run Federated Learning on Lung Cancer Dataset


In [ ]:
# Run federated learning with real cryptographic secure aggregation (Flower SecAgg+)
# Logs to LOG_DIR: fl_rounds.csv, fl_clients.csv, fl_eval.json, fl_summary.json, final_model.pth
import os

os.makedirs(LOG_DIR, exist_ok=True)

# Flower expects one --run-config string: 'k1=v1 k2=v2 ...' (not multiple args)
_run_config = (
    f'data-root="{DATA_ROOT}" '
    f'log-dir="{LOG_DIR}" '
    f'num-server-rounds={NUM_ROUNDS} '
    f'local-epochs={LOCAL_EPOCHS} '
    f'min-fit-clients={NUM_CLIENTS} '
    f'num-classes={num_classes} '
    f'dp-noise-fraction={DP_NOISE_FRACTION} '
    f'dp-noise-scale={DP_NOISE_SCALE} '
    'is-demo=false'
)

# Run in the notebook shell so all output and errors stream here
_cmd = f'cd "{PROJECT_DIR}" && PYTHONPATH="{PROJECT_DIR}" flwr run . -c \'{_run_config}\''
_exit_code = get_ipython().system(_cmd)

print(f"\n→ Results (fl_rounds.csv, fl_clients.csv, fl_eval.json, fl_summary.json, final_model.pth) are in: {LOG_DIR}")

# Quick check: list files if the run directory exists
if os.path.isdir(LOG_DIR):
    print("\nFiles in LOG_DIR:")
    for name in sorted(os.listdir(LOG_DIR)):
        print("  ", name)
else:
    print("\nLOG_DIR does not exist yet:", LOG_DIR)


## Plot Results (accuracy, F1, losses, timings, confusion matrix)


In [ ]:
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

log_dir = LOG_DIR
rounds_path = os.path.join(log_dir, "fl_rounds.csv")
clients_path = os.path.join(log_dir, "fl_clients.csv")
eval_path = os.path.join(log_dir, "fl_eval.json")
summary_path = os.path.join(log_dir, "fl_summary.json")

if not os.path.isfile(rounds_path) or not os.path.isfile(clients_path):
    print("No results yet. Run the previous cell (flwr run) first. If you already ran it, check LOG_DIR and that the SecAgg+ app wrote CSVs there.")
else:
    rounds_df = pd.read_csv(rounds_path)
    clients_df = pd.read_csv(clients_path)

    # Global test accuracy over rounds
    plt.figure(figsize=(10, 6))
    plt.plot(rounds_df["round"], rounds_df["global_test_acc"], marker="o", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Test Accuracy (%)")
    plt.title("Global Test Accuracy over FL Rounds (Lung Cancer)")
    plt.grid(True)
    plt.show()

    # F1 macro over rounds (if present)
    if "f1_macro" in rounds_df.columns:
        plt.figure(figsize=(10, 6))
        plt.plot(rounds_df["round"], rounds_df["f1_macro"], marker="o", linewidth=2, color="green")
        plt.xlabel("Round")
        plt.ylabel("F1 (macro)")
        plt.title("Global Test F1 (macro) over FL Rounds (Lung Cancer)")
        plt.grid(True)
        plt.show()

    # Inference and training time per round (if present)
    if "inference_time_sec" in rounds_df.columns and "training_time_sec" in rounds_df.columns:
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(rounds_df["round"], rounds_df["inference_time_sec"], marker="o", label="Inference time (s)", linewidth=2)
        ax.plot(rounds_df["round"], rounds_df["training_time_sec"], marker="s", label="Training time (s)", linewidth=2)
        ax.set_xlabel("Round")
        ax.set_ylabel("Time (s)")
        ax.set_title("Inference & Training Time per Round (Colon Cancer)")
        ax.legend()
        ax.grid(True)
        plt.tight_layout()
        plt.show()

    # Client training accuracies
    plt.figure(figsize=(12, 6))
    for cid in clients_df["client_id"].unique():
        d = clients_df[clients_df["client_id"] == cid]
        plt.plot(d["round"], d["train_acc"], marker="o", label=f"Client {cid}", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Training Accuracy (%)")
    plt.title("Client Training Accuracies over FL Rounds (Lung Cancer)")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Losses
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(rounds_df["round"], rounds_df["global_test_loss"], marker="o", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Test Loss")
    plt.title("Global Test Loss (Colon Cancer)")
    plt.grid(True)

    plt.subplot(1, 2, 2)
    for cid in clients_df["client_id"].unique():
        d = clients_df[clients_df["client_id"] == cid]
        plt.plot(d["round"], d["train_loss"], marker="o", label=f"Client {cid}", linewidth=2)
    plt.xlabel("Round")
    plt.ylabel("Training Loss")
    plt.title("Client Training Losses (Lung Cancer)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Final confusion matrix (from fl_eval.json, last round)
    if os.path.isfile(eval_path):
        with open(eval_path) as f:
            ev = json.load(f)
        rounds_ev = ev.get("rounds", [])
        class_names = ev.get("class_names")
        if not class_names and rounds_ev and "confusion_matrix" in rounds_ev[-1]:
            # Fallback generic names if class names are missing
            cm_tmp = np.array(rounds_ev[-1]["confusion_matrix"])
            class_names = [f"C{i}" for i in range(cm_tmp.shape[0])]

        if rounds_ev and "confusion_matrix" in rounds_ev[-1]:
            cm = np.array(rounds_ev[-1]["confusion_matrix"])
            plt.figure(figsize=(8, 6))
            plt.imshow(cm, interpolation="nearest", cmap="Blues")
            plt.colorbar()
            if class_names is not None:
                plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
                plt.yticks(np.arange(len(class_names)), class_names)
            plt.xlabel("Predicted")
            plt.ylabel("True")
            plt.title("Confusion Matrix (final round, Colon Cancer)")
            for i in range(cm.shape[0]):
                for j in range(cm.shape[1]):
                    plt.text(j, i, int(cm[i, j]), ha="center", va="center",
                             color="black" if cm[i, j] < cm.max() / 2 else "white")
            plt.tight_layout()
            plt.show()

    if os.path.isfile(summary_path):
        with open(summary_path) as f:
            s = json.load(f)
        print("Summary:", json.dumps(s, indent=2))


## Copy important result files (including trained model) back into the repo and push to GitHub

This cell copies only the **essential artifacts** from the run-specific directory into the repository under `results/`,
then stages, commits, and (optionally) pushes them to GitHub.

Artifacts copied:
- `fl_rounds.csv`
- `fl_clients.csv`
- `fl_eval.json`
- `config.json`
- `fl_summary.json`
- `final_model.pth` (final trained model weights, so you can re-use them without retraining)


In [ ]:
import os
import shutil

PROJECT_DIR = "/content/brain-tumor-classification-PIDL-FL"

# Run-specific log directory (source) and repo destination directory
SRC_DIR = LOG_DIR
DEST_DIR = os.path.join(
    PROJECT_DIR, "results", DATASET_NAME, f"{NUM_CLIENTS}_clients"
)

FILES = [
    "fl_rounds.csv",
    "fl_clients.csv",
    "fl_eval.json",
    "config.json",
    "fl_summary.json",
    "final_model.pth",
]

os.makedirs(DEST_DIR, exist_ok=True)
copied = []
for f in FILES:
    src = os.path.join(SRC_DIR, f)
    dst = os.path.join(DEST_DIR, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        copied.append(os.path.relpath(dst, PROJECT_DIR))
    else:
        print(f"Skipping {f} (not found in {SRC_DIR})")

if not copied:
    print("No result files to push. Run the FL cell first.")
else:
    print("Copied to repo (under results/):")
    for rel in copied:
        print("  ", rel)
    
    get_ipython().system(f'cd "{PROJECT_DIR}" && git config user.email "pulockkamol50@gmail.com"')
    get_ipython().system(f'cd "{PROJECT_DIR}" && git config user.name "PulockDas"')

    # Stage copied files
    for rel in copied:
        get_ipython().system(f'cd "{PROJECT_DIR}" && git add "{rel}"')
    get_ipython().system(f'cd "{PROJECT_DIR}" && git status')

    # Commit (ignore failure if nothing to commit)
    get_ipython().system(
        f'cd "{PROJECT_DIR}" && git commit -m "Add lung cancer FL result artifacts for {NUM_CLIENTS} clients"'
    )

    # Read token from environment (recommended: Colab Secrets -> env var)
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        print("\nNot pushing: missing GITHUB_TOKEN (Colab can't prompt for GitHub credentials).")
        print("Set GITHUB_TOKEN in the environment (e.g., Colab Secrets) and rerun this cell if you want to push.")
    else:
        origin_out = get_ipython().getoutput(
            f'cd "{PROJECT_DIR}" && git remote get-url origin'
        )
        origin_url = origin_out[0] if origin_out else ""
        if origin_url.startswith("https://"):
            push_url = origin_url.replace("https://", f"https://{token}@")
            get_ipython().system(f'cd "{PROJECT_DIR}" && git push "{push_url}" HEAD')
        else:
            # SSH remote
            get_ipython().system(f'cd "{PROJECT_DIR}" && git push origin HEAD')


In [ ]:
# Copy FL results to visible location for easy download
import os
import shutil

# Source: where your FL results are saved (use LOG_DIR from your notebook)
SRC_DIR = LOG_DIR  # This should already be set in your notebook

# Destination: visible location in Colab file browser
DOWNLOAD_DIR = "/content/FL_RESULTS"

# Create download directory
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# Files to copy
FILES = [
    "fl_rounds.csv",
    "fl_clients.csv",
    "fl_eval.json",
    "config.json",
    "fl_summary.json",
    "final_model.pth",
]

copied = []
for f in FILES:
    src = os.path.join(SRC_DIR, f)
    dst = os.path.join(DOWNLOAD_DIR, f)
    if os.path.isfile(src):
        shutil.copy2(src, dst)
        copied.append(f)
        print(f"✓ Copied: {f}")
    else:
        print(f"✗ Not found: {f}")

if copied:
    print(f"\n✅ All files copied to: {DOWNLOAD_DIR}")
    print("📁 You can now download them from Colab's file browser (left sidebar)")
    print(f"   Navigate to: {DOWNLOAD_DIR}")
    print("\n💡 To download: Right-click each file → Download")
else:
    print("❌ No files found. Check that LOG_DIR is set correctly.")